# 01 - Data Preparation

This notebook demonstrates the SO-SAFED data preprocessing pipeline:

1. Loading hourly PED admission data
2. Validating stationarity (ADF test)
3. Creating Darts TimeSeries objects
4. Generating temporal covariates
5. Aggregating into shift blocks for optimization

**Note:** Raw data files are not included in this repository (PHI protection).
You must supply your own CSV with columns: `date` (datetime), `apply_number` (hourly count).

In [ ]:
import sys
sys.path.insert(0, '../src')

from preprocessing.data_pipeline import DataPipeline

## 1. Load Data

Replace the path below with your own hourly admission CSV.

In [ ]:
DATA_PATH = "../data/hourly_admissions.csv"  # <-- Replace with your data path

pipeline = DataPipeline(data_path=DATA_PATH)
df = pipeline.load()
print(f"Loaded {len(df)} rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

## 2. Stationarity Check

In [ ]:
adf_result = pipeline.validate_stationarity()
print(f"ADF Statistic: {adf_result['adf_statistic']:.4f}")
print(f"P-value: {adf_result['p_value']:.2e}")
print(f"Stationary: {adf_result['is_stationary']}")

## 3. Create TimeSeries and Covariates

In [ ]:
raw_series, scaled_series, covariates = pipeline.prepare(freq='h')

print(f"Series length: {len(raw_series)}")
print(f"Covariates components: {covariates.n_components}")

## 4. Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

raw_series[-168:].plot(ax=axes[0], label='Last 7 days (raw)')
axes[0].set_title('Hourly PED Admissions')
axes[0].set_ylabel('Patients')

covariates[-168:].plot(ax=axes[1])
axes[1].set_title('Temporal Covariates')

plt.tight_layout()
plt.show()

## 5. Shift Block Aggregation

In [ ]:
shift_blocks = DataPipeline.aggregate_to_shift_blocks(df)
shift_blocks.head(9)